# GrooveTransformer Local GMD Prediction with Audio Playback

Run this notebook from the repository root. It picks a random local Groove MIDI Dataset MIDI file, converts it into the `1 x 32 x 3` input expected by the BaseVAE encoder, predicts a 9-voice drum pattern, and lets you listen to the MIDI-derived input and model output.


## 1. Optional: get the project files into Colab

If you opened this notebook from the repo, you can skip this cell. Otherwise upload or mount the project folder/checkpoint, then `%cd` into it. Set `GMD_MIDI_DIR` below to wherever your local GMD MIDI files are available.


In [1]:
# Example only - edit this for your own GitHub repository if needed.
# !git clone https://github.com/YOUR_USERNAME/GrooveTransformer-Lite.git
# %cd GrooveTransformer-Lite

# !pwd
# !ls -lh *.pth 2>/dev/null || true


## 2. Dependencies


In [2]:
import subprocess
import sys

try:
    import note_seq
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'note-seq'])
    import note_seq


## 3. Imports and model loader


In [3]:
import json
import os
import random
from pathlib import Path

import numpy as np
import torch
from IPython.display import Audio, display

from hvo_sequence.drum_mappings import ROLAND_REDUCED_MAPPING
from hvo_sequence.io_helpers import midi_to_hvo_sequence
from model import BaseVAE, MuteVAE, MuteGenreLatentVAE, MuteLatentGenreInputVAE


def load_model(model_path, model_class, params_dict=None, is_evaluating=True, device=None):
    model_path = Path(model_path)
    if not model_path.exists():
        raise FileNotFoundError(f'Model checkpoint not found: {model_path.resolve()}')

    try:
        loaded_dict = torch.load(model_path, map_location=device) if device is not None else torch.load(model_path)
    except Exception:
        loaded_dict = torch.load(model_path, map_location=torch.device('cpu'))

    if params_dict is None:
        if 'params' not in loaded_dict:
            raise ValueError('Checkpoint does not contain params. Pass params_dict as a dict or JSON path.')
        params_dict = loaded_dict['params']

    if isinstance(params_dict, str):
        with open(params_dict, 'r') as f:
            params_dict = json.load(f)

    model = model_class(params_dict)
    model.load_state_dict(loaded_dict['model_state_dict'])
    model.to(device or torch.device('cpu'))
    if is_evaluating:
        model.eval()
    return model


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


Could not import fluidsynth. AUDIO rendering will not work.


device(type='cuda')

## 4. Audio renderer

This lightweight renderer makes synthetic drum sounds directly with NumPy, so playback works in Colab without FluidSynth or a soundfont.


In [4]:
SR = 22050
BPM = 120
STEPS_PER_BEAT = 4

DRUM_NAMES = [
    'kick', 'snare', 'closed_hat', 'open_hat', 'low_tom',
    'mid_tom', 'high_tom', 'crash', 'ride'
]


def _exp_env(n, decay):
    return np.exp(-np.linspace(0.0, decay, n, endpoint=False))


def _noise(n, seed):
    rng = np.random.default_rng(seed)
    return rng.uniform(-1.0, 1.0, n)


def drum_voice(kind, velocity, sr=SR, seed=0):
    velocity = float(np.clip(velocity, 0.0, 1.0))

    if kind == 'kick':
        n = int(0.34 * sr)
        t = np.arange(n) / sr
        freq = 48 + 95 * np.exp(-t * 18)
        phase = 2 * np.pi * np.cumsum(freq) / sr
        return velocity * np.sin(phase) * _exp_env(n, 8.0)

    if kind == 'snare':
        n = int(0.24 * sr)
        t = np.arange(n) / sr
        tone = 0.35 * np.sin(2 * np.pi * 190 * t) * _exp_env(n, 9.0)
        buzz = _noise(n, seed) * _exp_env(n, 12.0)
        return velocity * (tone + 0.75 * buzz)

    if kind in {'closed_hat', 'ride'}:
        n = int((0.07 if kind == 'closed_hat' else 0.18) * sr)
        x = _noise(n, seed)
        x = np.concatenate([[0.0], np.diff(x)])
        return velocity * 0.45 * x * _exp_env(n, 18.0 if kind == 'closed_hat' else 8.0)

    if kind in {'open_hat', 'crash'}:
        n = int((0.42 if kind == 'open_hat' else 0.85) * sr)
        x = _noise(n, seed)
        x = np.concatenate([[0.0], np.diff(x)])
        return velocity * 0.35 * x * _exp_env(n, 5.5 if kind == 'open_hat' else 3.0)

    tom_freq = {'low_tom': 95, 'mid_tom': 130, 'high_tom': 175}.get(kind, 120)
    n = int(0.28 * sr)
    t = np.arange(n) / sr
    freq = tom_freq + 45 * np.exp(-t * 14)
    phase = 2 * np.pi * np.cumsum(freq) / sr
    return velocity * 0.75 * np.sin(phase) * _exp_env(n, 7.0)


def render_hvo_audio(hvo, bpm=BPM, sr=SR, steps_per_beat=STEPS_PER_BEAT, one_voice_name='kick'):
    hvo = np.asarray(hvo, dtype=np.float32)
    if hvo.ndim == 3:
        hvo = hvo[0]

    n_steps = hvo.shape[0]
    step_sec = 60.0 / bpm / steps_per_beat
    length = int((n_steps * step_sec + 1.2) * sr)
    audio = np.zeros(length, dtype=np.float32)

    if hvo.shape[1] == 3:
        hits = hvo[:, 0:1]
        velocities = hvo[:, 1:2]
        offsets = hvo[:, 2:3]
        names = [one_voice_name]
    else:
        n_voices = hvo.shape[1] // 3
        hits = hvo[:, :n_voices]
        velocities = hvo[:, n_voices:2 * n_voices]
        offsets = hvo[:, 2 * n_voices:3 * n_voices]
        names = DRUM_NAMES[:n_voices]

    event_id = 0
    for step in range(n_steps):
        for voice, name in enumerate(names):
            if hits[step, voice] <= 0.5:
                continue
            offset_sec = float(np.clip(offsets[step, voice], -0.5, 0.5)) * step_sec
            start = int((step * step_sec + offset_sec) * sr)
            start = max(0, min(start, length - 1))
            sample = drum_voice(name, velocities[step, voice], sr=sr, seed=event_id)
            end = min(length, start + len(sample))
            audio[start:end] += sample[:end - start]
            event_id += 1

    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = 0.92 * audio / peak
    return audio, sr


def play_hvo(label, hvo, **kwargs):
    print(label)
    audio, sr = render_hvo_audio(hvo, **kwargs)
    display(Audio(audio, rate=sr))


## 5. Load the model


In [5]:
beta = 0.2
model_path = f"base_vae_beta_{str(beta).replace('.', '_')}.pth"

model_base = load_model(model_path, BaseVAE, device=device)
print(f'Loaded {model_path} on {device}.')


Loaded base_vae_beta_0_2.pth on cuda.


## 6. Load a random local GMD MIDI and listen

This samples one `.mid`/`.midi` file from your local GMD folder, converts it to an `HVO_Sequence` with the project’s 9-voice Roland-reduced map, then flattens the selected 32-step segment into the model’s single-stream `h/v/o` encoder input.


In [21]:
GMD_MIDI_DIR = Path('/home/asantos-6/Documents/PhD/code/ADTS--Automatic-Drum-TapScription/data/raw/GMD/MIDIs')
GMD_RANDOM_SEED = random.randint(0, 2**31 - 1)
MODEL_STEPS = 32


def find_midi_files(midi_dir):
    midi_dir = Path(midi_dir).expanduser()
    if not midi_dir.exists():
        raise FileNotFoundError(f'GMD MIDI directory not found: {midi_dir}')
    midi_files = sorted(
        p for p in midi_dir.rglob('*')
        if p.is_file() and p.suffix.lower() in {'.mid', '.midi'}
    )
    if not midi_files:
        raise FileNotFoundError(f'No MIDI files found under: {midi_dir}')
    return midi_files


def choose_hvo_window(hvo, n_steps=MODEL_STEPS, seed=None):
    rng = np.random.default_rng(seed)
    n_voices = hvo.shape[1] // 3
    if hvo.shape[0] <= n_steps:
        padded = np.zeros((n_steps, hvo.shape[1]), dtype=np.float32)
        padded[:hvo.shape[0]] = hvo
        return padded, 0

    starts = np.arange(0, hvo.shape[0] - n_steps + 1)
    hit_counts = np.array([hvo[start:start + n_steps, :n_voices].sum() for start in starts])
    active_starts = starts[hit_counts > 0]
    start = int(rng.choice(active_starts if len(active_starts) else starts))
    return hvo[start:start + n_steps].astype(np.float32), start


def flatten_hvo_for_base_vae(hvo_9voice):
    n_voices = hvo_9voice.shape[1] // 3
    hits = hvo_9voice[:, :n_voices]
    velocities = hvo_9voice[:, n_voices:2 * n_voices] * hits
    offsets = hvo_9voice[:, 2 * n_voices:3 * n_voices] * hits

    max_voice = np.argmax(velocities, axis=1)
    step_ix = np.arange(hvo_9voice.shape[0])
    flat_hits = (hits.sum(axis=1) > 0).astype(np.float32)
    flat_velocities = velocities[step_ix, max_voice].astype(np.float32)
    flat_offsets = offsets[step_ix, max_voice].astype(np.float32)
    return np.stack([flat_hits, flat_velocities, flat_offsets], axis=-1)


rng = random.Random(GMD_RANDOM_SEED)
midi_files = find_midi_files(GMD_MIDI_DIR)
selected_midi_path = rng.choice(midi_files)

gmd_hvo_seq = midi_to_hvo_sequence(
    filename=str(selected_midi_path),
    drum_mapping=ROLAND_REDUCED_MAPPING,
    beat_division_factors=[STEPS_PER_BEAT],
)
if gmd_hvo_seq is None or gmd_hvo_seq.hvo is None:
    raise RuntimeError(f'Could not convert MIDI to HVO: {selected_midi_path}')

gmd_hvo_32, window_start = choose_hvo_window(gmd_hvo_seq.hvo, n_steps=MODEL_STEPS, seed=GMD_RANDOM_SEED)
input_groove_np = flatten_hvo_for_base_vae(gmd_hvo_32)
input_groove = torch.tensor(input_groove_np[None], dtype=torch.float32, device=device)

if gmd_hvo_seq.tempos:
    BPM = float(gmd_hvo_seq.tempos[0].qpm)

filename = str(selected_midi_path).split('/')[-1]  # Extract the filename from the path

print('Random seed:', GMD_RANDOM_SEED)
print('MIDI files found:', len(midi_files))
print('Selected MIDI file:', filename)
print('Window start step:', window_start)
print('Model input shape:', tuple(input_groove.shape))
print('Playback BPM:', BPM)

play_hvo('Original GMD 9-voice segment', gmd_hvo_32, bpm=BPM)
play_hvo('Model input: flattened GMD groove', input_groove_np, bpm=BPM, one_voice_name='closed_hat')


Random seed: 1292237526
MIDI files found: 786
Selected MIDI file: 1_latin-venezuelan-sangueo_101_beat_4-4.mid
Window start step: 372
Model input shape: (1, 32, 3)
Playback BPM: 101.00006901671382
Original GMD 9-voice segment


Model input: flattened GMD groove


## 7. Predict and listen to the model output


In [22]:
threshold = 0.5

with torch.no_grad():
    output_hvo, latent_z = model_base.predict(input_groove, threshold=threshold)

output_hvo_np = output_hvo.detach().cpu().numpy()
latent_z_np = latent_z.detach().cpu().numpy()

print('Output HVO shape:', output_hvo_np.shape)
print('Latent z shape:', latent_z_np.shape)
play_hvo('Model output', output_hvo_np, bpm=BPM)


Output HVO shape: (1, 32, 27)
Latent z shape: (1, 128)
Model output


## 8. Quick pattern inspection


In [23]:
hits = output_hvo_np[0, :, :9]
velocities = output_hvo_np[0, :, 9:18]

print('Hit counts by voice')
for name, count in zip(DRUM_NAMES, hits.sum(axis=0).astype(int)):
    print(f'{name:>10}: {count}')

print('\nFirst 8 steps, hit matrix:')
print(hits[:8].astype(int))


Hit counts by voice
      kick: 6
     snare: 19
closed_hat: 16
  open_hat: 0
   low_tom: 0
   mid_tom: 0
  high_tom: 0
     crash: 0
      ride: 5

First 8 steps, hit matrix:
[[1 0 1 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 0]
 [0 1 1 0 0 0 0 0 0]
 [1 0 0 0 0 0 0 0 0]
 [0 1 1 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0]
 [0 1 1 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0 0]]


## 9. Optional: save WAV files


In [24]:
from scipy.io import wavfile

gmd_audio, sr = render_hvo_audio(gmd_hvo_32, bpm=BPM)
input_audio, sr = render_hvo_audio(input_groove_np, bpm=BPM, one_voice_name='closed_hat')
output_audio, sr = render_hvo_audio(output_hvo_np, bpm=BPM)

wavfile.write('gmd_original_segment.wav', sr, (gmd_audio * 32767).astype(np.int16))
wavfile.write('input_groove.wav', sr, (input_audio * 32767).astype(np.int16))
wavfile.write('model_output.wav', sr, (output_audio * 32767).astype(np.int16))
print('Saved gmd_original_segment.wav, input_groove.wav, and model_output.wav')


Saved gmd_original_segment.wav, input_groove.wav, and model_output.wav
